# Unified Offset-Free MPC Baseline

This notebook replaces the split nominal and disturbance offset-free MPC workflows with a single `RUN_MODE` switch. It uses the shared baseline MPC runner and plotting modules so the notebook stays thin while preserving the current baseline-MPC experiment behavior and canonical saved pickle outputs.


In [ ]:
from pathlib import Path
import os
import pickle

RUN_MODE = "nominal"  # switch to "disturb" for disturbance mode
STYLE_PROFILE = "hybrid"  # choose from "hybrid", "paper", or "debug"


def resolve_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start] + list(start.parents):
        if (candidate / "Simulation").exists() and (candidate / "utils").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repo root from the current notebook working directory.")


REPO_ROOT = resolve_repo_root(Path.cwd())
os.chdir(REPO_ROOT)

RUN_PROFILE_MAP = {
    "nominal": {
        "use_disturbance": False,
        "result_prefix": "mpc_offsetfree_nominal_unified",
        "save_path": REPO_ROOT / "Data" / "mpc_results_nominal.pickle",
        "plot_start_episode": 1,
        "n_tests": 2,
        "set_points_len": 400,
        "test_cycle": [False, False],
        "nominal_qi": 108.0,
        "nominal_qs": 459.0,
        "nominal_ha": 1.05e6,
        "qi_change": 0.95,
        "qs_change": 1.05,
        "ha_change": 0.92,
    },
    "disturb": {
        "use_disturbance": True,
        "result_prefix": "mpc_offsetfree_disturb_unified",
        "save_path": REPO_ROOT / "Data" / "mpc_results_dist.pickle",
        "plot_start_episode": 1,
        "n_tests": 200,
        "set_points_len": 400,
        "test_cycle": [False, False],
        "nominal_qi": 108.0,
        "nominal_qs": 459.0,
        "nominal_ha": 1.05e6,
        "qi_change": 0.85,
        "qs_change": 1.3,
        "ha_change": 0.85,
    },
}

if RUN_MODE not in RUN_PROFILE_MAP:
    raise ValueError(f"RUN_MODE must be one of {list(RUN_PROFILE_MAP)}")
if STYLE_PROFILE not in {"hybrid", "paper", "debug"}:
    raise ValueError("STYLE_PROFILE must be 'hybrid', 'paper', or 'debug'.")

RUN_PROFILE = RUN_PROFILE_MAP[RUN_MODE]
RESULT_PREFIX = RUN_PROFILE["result_prefix"]
print(f"Repo root   : {REPO_ROOT}")
print(f"Run mode    : {RUN_MODE}")
print(f"Plot style  : {STYLE_PROFILE}")
print(f"Save target : {RUN_PROFILE['save_path']}")


In [ ]:
import numpy as np

from Simulation.mpc import MpcSolverGeneral
from Simulation.system_functions import PolymerCSTR
from utils.helpers import apply_min_max, load_and_prepare_system_data
from utils.mpc_baseline_runner import run_offsetfree_mpc
from utils.observer import compute_observer_gain
from utils.plotting import plot_baseline_mpc_results
from utils.rewards import make_reward_fn_relative_QR


In [ ]:
# First initiate the system
Ad = 2.142e17
Ed = 14897
Ap = 3.816e10
Ep = 3557
At = 4.50e12
Et = 843
fi = 0.6
m_delta_H_r = -6.99e4
hA = 1.05e6
rhocp = 1506
rhoccpc = 4043
Mm = 104.14
system_params = np.array([Ad, Ed, Ap, Ep, At, Et, fi, m_delta_H_r, hA, rhocp, rhoccpc, Mm])

CIf = 0.5888
CMf = 8.6981
Qi = 108.0
Qs = 459.0
Tf = 330.0
Tcf = 295.0
V = 3000.0
Vc = 3312.4
system_design_params = np.array([CIf, CMf, Qi, Qs, Tf, Tcf, V, Vc])

Qm_ss = 378.0
Qc_ss = 471.6
system_steady_state_inputs = np.array([Qc_ss, Qm_ss])

delta_t = 0.5
cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
steady_states = {"ss_inputs": cstr.ss_inputs, "y_ss": cstr.y_ss}

setpoint_y = np.array([[3.2, 321.0], [4.5, 325.0]])
u_min = np.array([71.6, 78.0])
u_max = np.array([870.0, 670.0])

system_data = load_and_prepare_system_data(
    steady_states=steady_states,
    setpoint_y=setpoint_y,
    u_min=u_min,
    u_max=u_max,
    data_dir=str(REPO_ROOT / "Data"),
)

A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]
data_min = system_data["data_min"]
data_max = system_data["data_max"]
min_max_dict = dict(system_data["min_max_dict"])

inputs_number = int(B_aug.shape[1])
y_sp_scenario = np.array([[4.5, 324.0], [3.4, 321.0]])
y_sp_scenario = apply_min_max(y_sp_scenario, data_min[inputs_number:], data_max[inputs_number:]) - apply_min_max(
    steady_states["y_ss"], data_min[inputs_number:], data_max[inputs_number:]
)


In [ ]:
poles = np.array([0.44619852, 0.33547649, 0.36380595, 0.70467118, 0.3562966, 0.42900673, 0.4228262, 0.96916776, 0.91230187])
L = compute_observer_gain(A_aug, C_aug, poles)

n_inputs = 2
predict_h = 9
cont_h = 3
Q1_penalty = 5.0
Q2_penalty = 1.0
R1_penalty = 1.0
R2_penalty = 1.0

u_ss = apply_min_max(steady_states["ss_inputs"], data_min[:n_inputs], data_max[:n_inputs])
b_min = apply_min_max(np.array([71.6, 78.0]), data_min[:n_inputs], data_max[:n_inputs]) - u_ss
b_max = apply_min_max(np.array([870.0, 670.0]), data_min[:n_inputs], data_max[:n_inputs]) - u_ss

MPC_obj = MpcSolverGeneral(
    A_aug,
    B_aug,
    C_aug,
    Q_out=np.array([Q1_penalty, Q2_penalty]),
    R_in=np.array([R1_penalty, R2_penalty]),
    NP=predict_h,
    NC=cont_h,
)

k_rel = np.array([0.003, 0.0003])
band_floor_phys = np.array([0.006, 0.07])
Q_diag = np.array([518.0, 90.0])
R_diag = np.array([90.0, 90.0])
reward_params, reward_fn = make_reward_fn_relative_QR(
    data_min,
    data_max,
    n_inputs=n_inputs,
    k_rel=k_rel,
    band_floor_phys=band_floor_phys,
    Q_diag=Q_diag,
    R_diag=R_diag,
    tau_frac=0.7,
    gamma_out=0.5,
    gamma_in=0.5,
    beta=7.0,
    gate="geom",
    lam_in=1.0,
    bonus_kind="exp",
    bonus_k=12.0,
    bonus_p=0.6,
    bonus_c=20.0,
)

print(reward_params)


In [ ]:
mpc_cfg = {
    "run_mode": RUN_MODE,
    "n_tests": RUN_PROFILE["n_tests"],
    "set_points_len": RUN_PROFILE["set_points_len"],
    "warm_start": 0,
    "test_cycle": RUN_PROFILE["test_cycle"],
    "predict_h": predict_h,
    "cont_h": cont_h,
    "Q1_penalty": Q1_penalty,
    "Q2_penalty": Q2_penalty,
    "R1_penalty": R1_penalty,
    "R2_penalty": R2_penalty,
    "nominal_qi": RUN_PROFILE["nominal_qi"],
    "nominal_qs": RUN_PROFILE["nominal_qs"],
    "nominal_ha": RUN_PROFILE["nominal_ha"],
    "qi_change": RUN_PROFILE["qi_change"],
    "qs_change": RUN_PROFILE["qs_change"],
    "ha_change": RUN_PROFILE["ha_change"],
    "b_min": b_min,
    "b_max": b_max,
}

runtime_ctx = {
    "system": PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t),
    "MPC_obj": MPC_obj,
    "steady_states": steady_states,
    "min_max_dict": min_max_dict,
    "data_min": data_min,
    "data_max": data_max,
    "A_aug": A_aug,
    "B_aug": B_aug,
    "C_aug": C_aug,
    "poles": poles,
    "L": L,
    "y_sp_scenario": y_sp_scenario,
    "reward_fn": reward_fn,
    "reward_params": reward_params,
}

result_bundle = run_offsetfree_mpc(mpc_cfg=mpc_cfg, runtime_ctx=runtime_ctx)
result_bundle["reward_params"] = reward_params
result_bundle["run_profile"] = RUN_PROFILE

print(f"nFE: {result_bundle['nFE']}")
print(f"Avg reward samples: {len(result_bundle['avg_rewards'])}")


In [ ]:
out_dir = plot_baseline_mpc_results(
    result_bundle=result_bundle,
    plot_cfg={
        "directory": REPO_ROOT / "Data",
        "prefix_name": RESULT_PREFIX,
        "start_episode": RUN_PROFILE["plot_start_episode"],
        "save_pdf": False,
        "style_profile": STYLE_PROFILE,
    },
)

legacy_payload = {
    "u_mpc": result_bundle["u"],
    "y_mpc": result_bundle["y"],
    "avg_rewards": result_bundle["avg_rewards"],
    "avg_rewards_mpc": result_bundle["avg_rewards_mpc"],
    "rewards_step": result_bundle["rewards_step"],
    "rewards_mpc": result_bundle["rewards_mpc"],
    "xhatdhat": result_bundle["xhatdhat"],
    "yhat": result_bundle["yhat"],
    "delta_y_storage": result_bundle["delta_y_storage"],
    "delta_u_storage": result_bundle["delta_u_storage"],
    "delat_u_storage": result_bundle["delta_u_storage"],
    "run_mode": result_bundle["run_mode"],
    "disturbance_profile": result_bundle["disturbance_profile"],
    "steady_states": result_bundle["steady_states"],
    "y_sp": result_bundle["y_sp"],
    "nFE": result_bundle["nFE"],
    "delta_t": result_bundle["delta_t"],
    "time_in_sub_episodes": result_bundle["time_in_sub_episodes"],
    "data_min": result_bundle["data_min"],
    "data_max": result_bundle["data_max"],
}

with open(RUN_PROFILE["save_path"], "wb") as file:
    pickle.dump(legacy_payload, file)

print(f"Plots saved to  : {out_dir}")
print(f"Baseline pickle : {RUN_PROFILE['save_path']}")
